# Cuadernillo Taller Unidad I - Electiva IV

Este cuadernillo desarrolla **punto por punto** el taller en formato de entrega.

Integrantes:

- David Santiago Lotero Rodríguez
- Gabriel Esteban Niño Avella
- Juan Pablo Muñoz Rojas

Estructura usada en cada punto:
1. Celda de **Enunciado**.
2. Celda de **Respuesta**.
3. Celda de **Codigo** (documentado linea a linea).

El archivo `.py` del proyecto se conserva sin cambios, como respaldo ejecutable.

## Preparacion general

Esta celda define utilidades comunes para que los puntos se ejecuten de forma incremental y ordenada.

In [21]:
# Importamos json para leer archivos en formato JSON.
import json
# Importamos defaultdict para acumular datos por clave facilmente.
from collections import defaultdict
# Importamos date para obtener la fecha actual en formato de cadena.
from datetime import date
# Importamos Path para construir rutas de archivo robustas.
from pathlib import Path
# Importamos pprint para mostrar estructuras complejas de forma legible.
from pprint import pprint

# Definimos ruta al archivo JSON externo dentro de la carpeta data.
RUTA_JSON = Path("data") / "Unidad1_Reto.json"

# Definimos una funcion para extraer tokens de nombre desde str, list o dict.
def extraer_tokens(valor):
    # Si el valor es texto, lo separamos por espacios y comas.
    if isinstance(valor, str):
        # Retornamos tokens limpios en minuscula.
        return [t.strip().lower() for t in valor.replace(",", " ").split() if t.strip()]
    # Si el valor es lista, acumulamos tokens de cada elemento.
    if isinstance(valor, list):
        # Inicializamos una lista temporal.
        salida = []
        # Recorremos cada elemento de la lista.
        for item in valor:
            # Extendemos la salida con los tokens de cada item.
            salida.extend(extraer_tokens(item))
        # Retornamos la lista acumulada.
        return salida
    # Si el valor es diccionario, recorremos claves en orden estable.
    if isinstance(valor, dict):
        # Inicializamos lista temporal.
        salida = []
        # Recorremos claves ordenadas alfabeticamente.
        for k in sorted(valor.keys()):
            # Extendemos salida con tokens del valor de cada clave.
            salida.extend(extraer_tokens(valor[k]))
        # Retornamos lista acumulada.
        return salida
    # Si no es tipo soportado, retornamos lista vacia.
    return []

# Definimos funcion para obtener primer y segundo nombre.
def obtener_nombres(estudiante):
    # Extraemos tokens del campo nombres.
    tokens = extraer_tokens(estudiante.get("nombres", ""))
    # Tomamos primer nombre si existe.
    primero = tokens[0] if len(tokens) >= 1 else ""
    # Tomamos segundo nombre si existe.
    segundo = tokens[1] if len(tokens) >= 2 else ""
    # Retornamos ambos valores.
    return primero, segundo

# Definimos funcion para obtener primer y segundo apellido.
def obtener_apellidos(estudiante):
    # Extraemos tokens del campo apellidos.
    tokens = extraer_tokens(estudiante.get("apellidos", ""))
    # Tomamos primer apellido si existe.
    primero = tokens[0] if len(tokens) >= 1 else ""
    # Tomamos segundo apellido si existe.
    segundo = tokens[1] if len(tokens) >= 2 else ""
    # Retornamos ambos valores.
    return primero, segundo

# Definimos funcion para calcular promedio por asignatura.
def promedio_por_asignatura(lista_estudiantes):
    # Creamos acumulador de listas por nombre de materia.
    acumulado = defaultdict(list)
    # Recorremos estudiantes.
    for est in lista_estudiantes:
        # Recorremos asignaturas del estudiante actual.
        for asig in est.get("asignaturas", []):
            # Leemos nombre de asignatura.
            nombre = str(asig.get("nombre", "")).strip()
            # Leemos nota de asignatura.
            nota = asig.get("nota")
            # Validamos nombre y tipo numerico de nota.
            if nombre and isinstance(nota, (int, float)):
                # Agregamos nota al acumulador de la materia.
                acumulado[nombre].append(float(nota))
    # Retornamos promedios por materia redondeados a 2 decimales.
    return {k: round(sum(v) / len(v), 2) for k, v in acumulado.items() if v}

# Definimos funcion para promedio por estudiante sin retiradas.
def promedio_estudiante_sin_retiradas(lista_estudiantes):
    # Inicializamos lista de resultados por estudiante.
    salida = []
    # Recorremos estudiantes.
    for est in lista_estudiantes:
        # Obtenemos nombres.
        pn, sn = obtener_nombres(est)
        # Obtenemos apellidos.
        pa, sa = obtener_apellidos(est)
        # Inicializamos lista de notas validas.
        notas = []
        # Recorremos asignaturas del estudiante.
        for asig in est.get("asignaturas", []):
            # Leemos bandera retirada normalizada.
            retirada = str(asig.get("retirada", "No")).strip().lower()
            # Leemos nota.
            nota = asig.get("nota")
            # Si no fue retirada y la nota es valida, la acumulamos.
            if retirada == "no" and isinstance(nota, (int, float)):
                notas.append(float(nota))
        # Calculamos promedio o 0.0 si no hay notas.
        promedio = round(sum(notas) / len(notas), 2) if notas else 0.0
        # Agregamos registro del estudiante.
        salida.append({
            "codigo": est.get("codigo", ""),
            "documento": est.get("documento", ""),
            "primer_nombre": pn,
            "segundo_nombre": sn,
            "primer_apellido": pa,
            "segundo_apellido": sa,
            "promedio": promedio,
        })
    # Ordenamos por apellido y nombre.
    return sorted(salida, key=lambda r: (r["primer_apellido"], r["segundo_apellido"], r["primer_nombre"]))

# Definimos funcion para correo institucional.
def correo_institucional(estudiante, dominio="uptc.edu.co"):
    # Obtenemos nombres del estudiante.
    pn, sn = obtener_nombres(estudiante)
    # Obtenemos apellidos del estudiante.
    pa, sa = obtener_apellidos(estudiante)
    # Convertimos documento a texto.
    documento = str(estudiante.get("documento", "")).strip()
    # Tomamos los dos ultimos digitos del documento.
    ult2 = documento[-2:] if len(documento) >= 2 else documento
    # Si existe segundo nombre, aplicamos regla de dos nombres.
    if sn:
        # Construimos usuario con iniciales y primer apellido.
        usuario = f"{pn[:1]}{sn[:1]}.{pa}{ult2}"
    # Si no existe segundo nombre, aplicamos regla de un nombre.
    else:
        # Construimos usuario con inicial nombre + inicial apellido + segundo apellido.
        usuario = f"{pn[:1]}{pa[:1]}.{sa}{ult2}"
    # Retornamos correo en minuscula.
    return f"{usuario.lower()}@{dominio}"

# Definimos funcion para vocales.
def es_vocal(c):
    # Retornamos False si no es una sola letra.
    if not c or len(c) != 1 or not c.isalpha():
        return False
    # Retornamos True si esta en vocales.
    return c.lower() in "aeiou"

# Definimos funcion para ecuacion cuadratica.
def resolver_cuadratica(a, b, c):
    # Validamos que a sea diferente de cero.
    if a == 0:
        raise ValueError("a no puede ser 0")
    # Calculamos discriminante.
    d = b**2 - 4 * a * c
    # Calculamos raiz del discriminante (real o compleja).
    raiz = d**0.5 if d >= 0 else complex(0, (-d) ** 0.5)
    # Calculamos primera solucion.
    x1 = (-b + raiz) / (2 * a)
    # Calculamos segunda solucion.
    x2 = (-b - raiz) / (2 * a)
    # Retornamos ambas soluciones.
    return x1, x2

# Definimos funcion para histograma.
def histogram(valores, simbolo="H"):
    # Recorremos cada valor.
    for n in valores:
        # Si es entero positivo, imprimimos simbolo repetido.
        if isinstance(n, int) and n > 0:
            print(simbolo * n)
        # Si no, imprimimos linea vacia.
        else:
            print("")

# Definimos funcion de metodos de cadena.
def demo_cadenas():
    # Definimos texto base de prueba.
    texto = "  gestion de datos con python  "
    # Retornamos resultados de metodos solicitados.
    return {
        "replace": texto.replace("python", "analitica"),
        "find": texto.find("datos"),
        "count": texto.count("o"),
        "capitalize": texto.strip().capitalize(),
        "title": texto.strip().title(),
        "rstrip": texto.rstrip(),
        "index": texto.index("de"),
        "casefold": "GESTION".casefold(),
    }

# Definimos funcion de operaciones sobre frase.
def operaciones_frase(frase):
    # Partimos frase en palabras.
    palabras = frase.split()
    # Convertimos frase en lista de caracteres.
    letras = list(frase)
    # Reemplazamos Gestion por Analitica con y sin tilde.
    reemplazo = frase.replace("Gestion", "Analitica").replace("Gestión", "Analítica")
    # Unimos palabras con guion.
    guion_palabras = "-".join(palabras)
    # Unimos letras con guion.
    guion_letras = "-".join(letras)
    # Retornamos cuatro salidas solicitadas.
    return palabras, reemplazo, guion_letras, guion_palabras

# Definimos funcion para extraer mes actual.
def mes_actual():
    # Obtenemos fecha actual en formato ISO.
    fecha = date.today().isoformat()
    # Extraemos el mes en posicion 5 a 7.
    mes = fecha[5:7]
    # Retornamos texto formateado.
    return f"Mes {mes}"

# Definimos funcion para imprimir empleados.
def imprimir_empleados(empleados):
    # Desempaquetamos estructura esperada.
    nombres, edades, pesos = empleados
    # Hallamos minimo comun de longitudes.
    total = min(len(nombres), len(edades), len(pesos))
    # Recorremos indices validos.
    for i in range(total):
        # Imprimimos linea con formato de enunciado.
        print(f"Empleado # {i + 1}: Nombre: {nombres[i]},  Edad: {edades[i]}, Peso: {pesos[i]}")

# Definimos funcion de ejemplos de listas.
def ejemplos_listas():
    # Creamos lista base.
    base = [10, 20, 30]
    # Obtenemos copia superficial.
    copia = base.copy()
    # Creamos lista para remove.
    l_remove = [1, 2, 3, 2]
    # Removemos primera ocurrencia de 2.
    l_remove.remove(2)
    # Creamos lista para del.
    l_del = ["a", "b", "c", "d"]
    # Eliminamos elemento en indice 1.
    del l_del[1]
    # Creamos lista para clear.
    l_clear = [1, 2, 3]
    # Vaciamos la lista.
    l_clear.clear()
    # Evaluamos operador in.
    existe = 20 in base
    # Creamos lista para append.
    l_append = [1, 2]
    # Append agrega un solo elemento (lista anidada).
    l_append.append([3, 4])
    # Creamos lista para extend.
    l_extend = [1, 2]
    # Extend agrega cada elemento por separado.
    l_extend.extend([3, 4])
    # Retornamos resumen de resultados.
    return {
        "copy": copia,
        "remove": l_remove,
        "del": l_del,
        "clear": l_clear,
        "in": existe,
        "append": l_append,
        "extend": l_extend,
    }

# Definimos funcion para detectar duplicados.
def tiene_duplicados(lista):
    # Inicializamos conjunto de vistos.
    vistos = set()
    # Recorremos lista original.
    for e in lista:
        # Si ya estaba visto, hay duplicado.
        if e in vistos:
            return True
        # Si no, lo marcamos como visto.
        vistos.add(e)
    # Si termina sin repetidos, retornamos False.
    return False

# Definimos funcion para eliminar duplicados preservando orden.
def quitar_duplicados(lista):
    # Inicializamos conjunto de vistos.
    vistos = set()
    # Inicializamos salida.
    salida = []
    # Recorremos elementos de entrada.
    for e in lista:
        # Si no fue visto antes, lo conservamos.
        if e not in vistos:
            # Marcamos elemento como visto.
            vistos.add(e)
            # Agregamos elemento a la salida.
            salida.append(e)
    # Retornamos lista sin duplicados.
    return salida

# Definimos funcion de diferencia entre set y frozenset.
def diferencia_set_frozenset():
    # Retornamos descripcion breve de ambas estructuras.
    return {
        "set": "Mutable; permite add/remove/update",
        "frozenset": "Inmutable; no admite cambios",
    }

# Definimos funcion con ejemplos de operaciones de conjuntos.
def demo_conjuntos():
    # Definimos conjuntos base.
    a = {1, 2, 3, 4}
    # Definimos segundo conjunto base.
    b = {3, 4, 5}
    # Probamos intersection_update en copia.
    inter_upd = a.copy()
    # Actualizamos dejando solo interseccion.
    inter_upd.intersection_update(b)
    # Probamos isdisjoint.
    disjoint = {1, 2}.isdisjoint({3, 4})
    # Probamos issubset.
    subset = {1, 2}.issubset({1, 2, 3})
    # Probamos issuperset.
    superset = {1, 2, 3}.issuperset({1, 2})
    # Probamos pop.
    c_pop = {7, 8, 9}
    # Extraemos un valor.
    extraido = c_pop.pop()
    # Probamos remove.
    c_remove = {10, 11, 12}
    # Removemos un elemento especifico.
    c_remove.remove(11)
    # Probamos symmetric_difference.
    sim_diff = {1, 2, 3}.symmetric_difference({3, 4, 5})
    # Probamos symmetric_difference_update.
    sim_diff_upd = {1, 2, 3}
    # Actualizamos set en sitio.
    sim_diff_upd.symmetric_difference_update({3, 4, 5})
    # Probamos union.
    union = {1, 2}.union({2, 3, 4})
    # Probamos update.
    upd = {1, 2}
    # Actualizamos con elementos nuevos.
    upd.update({2, 3, 4})
    # Retornamos todos los resultados.
    return {
        "intersection_update": inter_upd,
        "isdisjoint": disjoint,
        "issubset": subset,
        "issuperset": superset,
        "pop": {"valor_extraido": extraido, "set_restante": c_pop},
        "remove": c_remove,
        "symmetric_difference": sim_diff,
        "symmetric_difference_update": sim_diff_upd,
        "union": union,
        "update": upd,
    }

# Definimos funcion para agrupar palabras por inicial.
def agrupar_inicial(words):
    # Creamos diccionario de listas por defecto.
    grupos = defaultdict(list)
    # Recorremos cada palabra.
    for w in words:
        # Validamos palabra no vacia.
        if w:
            # Agregamos palabra al grupo de su inicial.
            grupos[w[0].lower()].append(w)
    # Convertimos defaultdict a dict comun.
    return dict(grupos)

# Definimos funcion para contar vocales y consonantes.
def contar_vocales_consonantes(texto):
    # Definimos vocales permitidas.
    vocales = "aeiou"
    # Inicializamos conteo de vocales en cero.
    conteo = {v: 0 for v in vocales}
    # Inicializamos contador de consonantes.
    consonantes = 0
    # Recorremos texto en minuscula.
    for ch in texto.lower():
        # Solo evaluamos caracteres alfabeticos.
        if ch.isalpha():
            # Si es vocal incrementamos su contador.
            if ch in vocales:
                conteo[ch] += 1
            # Si no es vocal, sumamos una consonante.
            else:
                consonantes += 1
    # Guardamos total de consonantes.
    conteo["Consonantes"] = consonantes
    # Retornamos resultado final.
    return conteo

# Definimos funcion para convertir cadena de clientes a diccionario anidado.
def clientes_a_diccionario(data):
    # Separamos lineas no vacias.
    lineas = [ln.strip() for ln in data.split("\n") if ln.strip()]
    # Si no hay lineas, retornamos vacio.
    if not lineas:
        return {}
    # Leemos encabezados desde primera linea.
    headers = [h.strip() for h in lineas[0].split(";")]
    # Inicializamos resultado.
    salida = {}
    # Recorremos lineas de datos.
    for ln in lineas[1:]:
        # Separamos campos por ;
        campos = [c.strip() for c in ln.split(";")]
        # Validamos cantidad de campos.
        if len(campos) != len(headers):
            continue
        # Construimos registro de fila.
        registro = dict(zip(headers, campos))
        # Extraemos id para llave principal.
        id_cliente = registro.pop("id")
        # Guardamos registro sin id en salida.
        salida[id_cliente] = registro
    # Retornamos diccionario final.
    return salida

# Mensaje de preparacion completada.
print("Preparacion lista. Puedes ejecutar cada punto en orden.")

Preparacion lista. Puedes ejecutar cada punto en orden.


## Actividad 1 - Enunciado

Cargar el archivo `Unidad1_Reto.json` al entorno Python.

## Actividad 1 - Respuesta

Se carga el JSON desde `data/Unidad1_Reto.json` y se valida la cantidad de registros leidos.

In [22]:
# Abrimos el archivo JSON usando la ruta definida en preparacion.
with open(RUTA_JSON, "r", encoding="utf-8") as f:
    # Cargamos los estudiantes en una variable global reutilizable.
    estudiantes = json.load(f)
# Imprimimos la ruta utilizada para trazabilidad.
print("Ruta cargada:", RUTA_JSON)
# Imprimimos la cantidad de estudiantes leidos.
print("Total estudiantes:", len(estudiantes))

Ruta cargada: data\Unidad1_Reto.json
Total estudiantes: 4


## Actividad 2 - Enunciado

Calcular la nota promedio por asignatura.

## Actividad 2 - Respuesta

Se usa una coleccion tipo diccionario donde la llave es la asignatura y el valor es su promedio.

In [23]:
# Calculamos el promedio por asignatura con la funcion de apoyo.
prom_asig = promedio_por_asignatura(estudiantes)
# Mostramos un encabezado para la salida.
print("Promedio por asignatura:")
# Imprimimos el diccionario de resultados de forma legible.
pprint(prom_asig)

Promedio por asignatura:
{'Calculo 1': 3.47,
 'Fisica II': 3.03,
 'Introduccion a la Ingenieria': 3.3,
 'Programacion I': 3.05}


## Actividad 3 - Enunciado

Calcular la nota promedio por estudiante en asignaturas no retiradas y ordenar por apellido.

## Actividad 3 - Respuesta

Se filtran materias con `retirada = No`, se calcula promedio individual y se retorna una lista ordenada por apellidos.

In [24]:
# Calculamos promedios por estudiante usando solo asignaturas no retiradas.
prom_est = promedio_estudiante_sin_retiradas(estudiantes)
# Mostramos encabezado del resultado.
print("Promedio por estudiante (no retiradas, ordenado por apellido):")
# Mostramos la coleccion resultante.
pprint(prom_est)

Promedio por estudiante (no retiradas, ordenado por apellido):
[{'codigo': '20220178002',
  'documento': 9345787,
  'primer_apellido': 'flechas',
  'primer_nombre': 'pedro',
  'promedio': 3.05,
  'segundo_apellido': 'salamanca',
  'segundo_nombre': 'luis'},
 {'codigo': '20210278001',
  'documento': 123,
  'primer_apellido': 'perez',
  'primer_nombre': 'juan',
  'promedio': 2.85,
  'segundo_apellido': 'rubiano',
  'segundo_nombre': 'carlos'},
 {'codigo': '20210294001',
  'documento': 6879,
  'primer_apellido': 'ramirez',
  'primer_nombre': 'luis',
  'promedio': 3.17,
  'segundo_apellido': 'moreno',
  'segundo_nombre': ''},
 {'codigo': '20220178001',
  'documento': 879345,
  'primer_apellido': 'salinas',
  'primer_nombre': 'laura',
  'promedio': 3.8,
  'segundo_apellido': 'fonseca',
  'segundo_nombre': 'camila'}]


## Actividad 4 - Enunciado

Generar una lista de estudiantes con su correo institucional segun la regla dada.

## Actividad 4 - Respuesta

Se genera un usuario con iniciales, apellidos y dos ultimos digitos del documento; luego se concatena el dominio institucional.

In [25]:
# Inicializamos una lista para guardar la salida final.
lista_correos = []
# Recorremos cada estudiante para construir su correo.
for est in estudiantes:
    # Obtenemos nombres y apellidos normalizados.
    pn, sn = obtener_nombres(est)
    pa, sa = obtener_apellidos(est)
    # Construimos nombre completo con formato titulo.
    nombre = " ".join([x for x in [pn, sn, pa, sa] if x]).title()
    # Agregamos registro de salida con codigo, nombre y correo.
    lista_correos.append({"codigo": est.get("codigo", ""), "nombre": nombre, "correo": correo_institucional(est)})
# Mostramos la lista de correos institucionales.
pprint(lista_correos)

[{'codigo': '20210278001',
  'correo': 'jc.perez23@uptc.edu.co',
  'nombre': 'Juan Carlos Perez Rubiano'},
 {'codigo': '20210294001',
  'correo': 'lr.moreno79@uptc.edu.co',
  'nombre': 'Luis Ramirez Moreno'},
 {'codigo': '20220178001',
  'correo': 'lc.salinas45@uptc.edu.co',
  'nombre': 'Laura Camila Salinas Fonseca'},
 {'codigo': '20220178002',
  'correo': 'pl.flechas87@uptc.edu.co',
  'nombre': 'Pedro Luis Flechas Salamanca'}]


# Actividad 5 - Cuadernillos

A continuacion se desarrolla cada subpunto (5.1 a 5.15) en formato Enunciado, Respuesta y Codigo.

## 5.1 - Enunciado

Escribir una funcion que reciba un caracter y evale si es vocal o consonante; cuando sea vocal retorna True.

## 5.1 - Respuesta

Se implementa `es_vocal` y se prueba con dos entradas para evidenciar ambos casos.

In [26]:
# Probamos la funcion con una vocal.
print("Entrada: 'a' ->", es_vocal("a"))
# Probamos la funcion con una consonante.
print("Entrada: 'b' ->", es_vocal("b"))

Entrada: 'a' -> True
Entrada: 'b' -> False


## 5.2 - Enunciado

Convertir el ejercicio de ecuacion cuadratica a funcion.

## 5.2 - Respuesta

Se llama la funcion `resolver_cuadratica(a, b, c)` y se imprimen las dos raices.

In [27]:
# Definimos coeficientes del ejemplo.
a, b, c = 1, -5, 6
# Ejecutamos la funcion cuadratica.
raices = resolver_cuadratica(a, b, c)
# Mostramos las raices obtenidas.
print("Raices:", raices)

Raices: (3.0, 2.0)


## 5.3 - Enunciado

Hacer una funcion que reciba una lista de numeros e imprima un histograma.

## 5.3 - Respuesta

Se usa `histogram([3, 5, 1])` para imprimir lineas de H de acuerdo con cada valor.

In [28]:
# Definimos lista de ejemplo para el histograma.
valores = [3, 5, 1]
# Mostramos titulo de salida.
print("Histograma:")
# Ejecutamos la impresion del histograma.
histogram(valores)

Histograma:
HHH
HHHHH
H


## 5.4 - Enunciado

Explicar con ejemplo: replace, find, count, capitalize, title, rstrip, index, casefold.

## 5.4 - Respuesta

Se explican y muestran ejemplos de los métodos solicitados:

- `replace`: Reemplaza subcadenas. Ejemplo: `texto.replace("python", "analitica")` -> '  gestion de datos con analitica  '
- `find`: Devuelve el índice de la primera aparición o -1 si no existe. Ejemplo: `texto.find("datos")` -> 13
- `count`: Cuenta ocurrencias. Ejemplo: `texto.count("o")` -> 4
- `capitalize`: Capitaliza la primera letra del texto (después de strip). Ejemplo: `texto.strip().capitalize()` -> 'Gestion de datos con python'
- `title`: Pone en formato Title cada palabra. Ejemplo: `texto.strip().title()` -> 'Gestion De Datos Con Python'
- `rstrip`: Elimina espacios a la derecha. Ejemplo: `texto.rstrip()` -> '  gestion de datos con python'
- `index`: Como `find` pero lanza `ValueError` si no existe. Ejemplo: `texto.index("de")` -> 10
- `casefold`: Normaliza texto para comparaciones insensibles a mayúsculas/acentos. Ejemplo: `"GESTION".casefold()` -> 'gestion'

In [29]:
# Ejecutamos la funcion de demostracion de metodos de cadena.
resultado_54 = demo_cadenas()
# Mostramos titulo de salida.
print("Resultados metodos de cadena:")
# Imprimimos resultados en formato legible.
pprint(resultado_54)

Resultados metodos de cadena:
{'capitalize': 'Gestion de datos con python',
 'casefold': 'gestion',
 'count': 4,
 'find': 13,
 'index': 10,
 'replace': '  gestion de datos con analitica  ',
 'rstrip': '  gestion de datos con python',
 'title': 'Gestion De Datos Con Python'}


## 5.5 - Enunciado

Dividir la frase en palabras y letras, reemplazar texto y unir con guiones.

## 5.5 - Respuesta

Se usa `split`, `replace` y `join` para generar exactamente las cuatro salidas esperadas.

In [30]:
# Definimos la frase de entrada solicitada.
frase = "Gestión de Datos"
# Ejecutamos transformaciones definidas en la funcion.
s1, s2, s3, s4 = operaciones_frase(frase)
# Imprimimos salida de palabras.
print(s1)
# Imprimimos salida de reemplazo.
print(s2)
# Imprimimos salida de letras con guiones.
print(s3)
# Imprimimos salida de frase unida con guiones.
print(s4)

['Gestión', 'de', 'Datos']
Analítica de Datos
G-e-s-t-i-ó-n- -d-e- -D-a-t-o-s
Gestión-de-Datos


## 5.6 - Enunciado

Obtener la fecha actual y extraer el mes desde su representacion en cadena.

## 5.6 - Respuesta

Se toma la fecha ISO actual y se devuelve el mes en formato `Mes XX`.

In [31]:
# Ejecutamos la funcion que retorna el mes actual.
resultado_56 = mes_actual()
# Imprimimos el resultado en pantalla.
print(resultado_56)

Mes 04


## 5.7 - Enunciado

Mostrar empleados con el formato especificado a partir de la lista `[nombres, edades, peso]`.

## 5.7 - Respuesta

Se recorre la estructura por indice y se imprime cada registro con la plantilla exacta solicitada.

In [32]:
# Definimos nombres de empleados.
nombres = ["Luis", "Pedro", "Lucia"]
# Definimos edades de empleados.
edades = [20, 18, 30]
# Definimos pesos de empleados.
pesos = [55.6, 60, 65.8]
# Construimos lista compuesta de empleados.
empleados = [nombres, edades, pesos]
# Ejecutamos impresion con formato.
imprimir_empleados(empleados)

Empleado # 1: Nombre: Luis,  Edad: 20, Peso: 55.6
Empleado # 2: Nombre: Pedro,  Edad: 18, Peso: 60
Empleado # 3: Nombre: Lucia,  Edad: 30, Peso: 65.8


## 5.8 - Enunciado

Consultar copy, remove, del, clear, in y diferencia entre append y extend.

## 5.8 - Respuesta

Se describen los métodos y operadores pedidos con ejemplos y resultado esperado:

- `copy`: `base = [10,20,30]; copia = base.copy()` -> copia independiente: [10, 20, 30]
- `remove`: `l = [1,2,3,2]; l.remove(2)` -> [1, 3, 2] (elimina la PRIMERA aparición de 2)
- `del`: `l = ['a','b','c','d']; del l[1]` -> ['a', 'c', 'd'] (elimina por índice)
- `clear`: `l = [1,2,3]; l.clear()` -> [] (vacía la lista)
- `in`: operador de pertenencia. Ejemplo: `20 in base` -> True
- `append` vs `extend`:
  - `append`: agrega el argumento como un solo elemento. Ejemplo: `a=[1,2]; a.append([3,4])` -> [1,2,[3,4]]
  - `extend`: agrega cada elemento del iterable. Ejemplo: `b=[1,2]; b.extend([3,4])` -> [1,2,3,4]
- Nota: `append` es útil para añadir objetos individuales; `extend` para concatenar iterables.

In [33]:
# Ejecutamos la funcion de ejemplos de listas.
resultado_58 = ejemplos_listas()
# Mostramos resultados de cada operacion.
pprint(resultado_58)

{'append': [1, 2, [3, 4]],
 'clear': [],
 'copy': [10, 20, 30],
 'del': ['a', 'c', 'd'],
 'extend': [1, 2, 3, 4],
 'in': True,
 'remove': [1, 3, 2]}


## 5.9 - Enunciado

Crear funcion que indique si una lista tiene elementos duplicados sin modificarla.

## 5.9 - Respuesta

Se recorre la lista con un conjunto de vistos; si aparece repetido retorna True.

In [34]:
# Evaluamos una lista con duplicados.
print(tiene_duplicados([1, 2, 3, 2]))
# Evaluamos una lista sin duplicados.
print(tiene_duplicados([1, 2, 3]))

True
False


## 5.10 - Enunciado

Hacer una funcion que elimine duplicados de una lista.

## 5.10 - Respuesta

Se conserva el orden original y se mantiene solo la primera aparicion de cada elemento.

In [35]:
# Definimos lista de entrada con elementos repetidos.
entrada_510 = [1, 2, 2, 3, 1, 4]
# Ejecutamos eliminacion de duplicados.
salida_510 = quitar_duplicados(entrada_510)
# Mostramos resultado final.
print(salida_510)

[1, 2, 3, 4]


## 5.11 - Enunciado

Consultar la diferencia entre set y frozenset.

## 5.11 - Respuesta

Explicación y ejemplos:

- `set` (conjunto mutable): permite operaciones en sitio como `add`, `remove`, `pop`. Ejemplo: `s = {1,2}; s.add(3)` -> {1,2,3}
- `frozenset` (conjunto inmutable): no permite modificación en sitio. Ejemplo: `fs = frozenset({1,2})`; `fs.add(3)` -> AttributeError (no existe `add`), pero `fs.union({3})` -> frozenset({1,2,3}) devuelve un nuevo frozenset
- Usos: `set` cuando se necesita cambiar el conjunto; `frozenset` para claves inmutables en diccionarios o cuando se requiere hashabilidad.

In [36]:
# Ejecutamos funcion de diferencia conceptual.
resultado_511 = diferencia_set_frozenset()
# Imprimimos comparacion entre ambas estructuras.
pprint(resultado_511)

{'frozenset': 'Inmutable; no admite cambios',
 'set': 'Mutable; permite add/remove/update'}


## 5.12 - Enunciado

Usar: intersection_update, isdisjoint, issubset, issuperset, pop, remove, symmetric_difference, symmetric_difference_update, union, update.

## 5.12 - Respuesta

Se ejecuta un ejemplo por cada metodo y se retorna en una sola estructura de resultados.

In [37]:
# Ejecutamos demostracion completa de operaciones con conjuntos.
resultado_512 = demo_conjuntos()
# Imprimimos todos los resultados en formato legible.
pprint(resultado_512)

{'intersection_update': {3, 4},
 'isdisjoint': True,
 'issubset': True,
 'issuperset': True,
 'pop': {'set_restante': {9, 7}, 'valor_extraido': 8},
 'remove': {10, 12},
 'symmetric_difference': {1, 2, 4, 5},
 'symmetric_difference_update': {1, 2, 4, 5},
 'union': {1, 2, 3, 4},
 'update': {1, 2, 3, 4}}


## 5.13 - Enunciado

Crear un diccionario desde una lista agrupando palabras que inician con la misma letra.

## 5.13 - Respuesta

Se recorre la lista y se inserta cada palabra en la llave de su inicial.

In [38]:
# Definimos lista de palabras de entrada.
words = ["apple", "bat", "bar", "atom", "book", "cat"]
# Ejecutamos agrupacion por letra inicial.
resultado_513 = agrupar_inicial(words)
# Mostramos el diccionario de grupos.
pprint(resultado_513)

{'a': ['apple', 'atom'], 'b': ['bat', 'bar', 'book'], 'c': ['cat']}


## 5.14 - Enunciado

Recibir una cadena y devolver conteo de cada vocal y de consonantes.

## 5.14 - Respuesta

Se recorre la cadena caracter por caracter, se acumulan vocales por llave y consonantes en un contador.

In [39]:
# Definimos cadena de entrada para conteo.
cad = "Gestion de Datos"
# Ejecutamos funcion de conteo.
resultado_514 = contar_vocales_consonantes(cad)
# Mostramos diccionario resultante.
pprint(resultado_514)

{'Consonantes': 8, 'a': 1, 'e': 2, 'i': 1, 'o': 2, 'u': 0}


## 5.15 - Enunciado

Convertir cadena de clientes con separadores `;` y saltos de linea en diccionario anidado por ID.

## 5.15 - Respuesta

Se parsea la cabecera, se procesa cada fila y se guarda cada cliente en la llave de su ID.

In [40]:
# Definimos la cadena de entrada de clientes.
cadena = (
    "id;nombre;correo;movil;salario\n"
    "3412;Pepe Perez;pepeperez@yahoo.com;300281234;150000\n"
    "45342;Maria Melo;mariamelo@yahoo.com;315434223;300000\n"
    "5673321;Fernando Jimenez;ferjim@gmail.com;312342234;230000\n"
    "4545231;Carlos Cardenas;carloscardenas@hotmail.com;3156754323;345000"
)
# Convertimos cadena a diccionario anidado.
resultado_515 = clientes_a_diccionario(cadena)
# Imprimimos resultado final.
pprint(resultado_515)

{'3412': {'correo': 'pepeperez@yahoo.com',
          'movil': '300281234',
          'nombre': 'Pepe Perez',
          'salario': '150000'},
 '45342': {'correo': 'mariamelo@yahoo.com',
           'movil': '315434223',
           'nombre': 'Maria Melo',
           'salario': '300000'},
 '4545231': {'correo': 'carloscardenas@hotmail.com',
             'movil': '3156754323',
             'nombre': 'Carlos Cardenas',
             'salario': '345000'},
 '5673321': {'correo': 'ferjim@gmail.com',
             'movil': '312342234',
             'nombre': 'Fernando Jimenez',
             'salario': '230000'}}
